# Exercise 3 — parse_ops_step and run_ops_step

`parse_ops_step` reads one ReAct turn from the LLM's raw text and returns a structured dict.  It handles two cases: an action step (Thought/Action/Input) and a final-answer step (Thought/Final Answer).  `run_ops_step` calls the LLM once and passes the result to the parser.

In [ ]:
import json
from dataclasses import dataclass, field

def call_llm(messages, llm_fn=None):
    if llm_fn is not None:
        return str(llm_fn(messages))
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

def safe_parse_json(text):
    text = str(text)
    start = text.find("{")
    end   = text.rfind("}") + 1
    if start == -1 or end == 0:
        return None
    try:
        return json.loads(text[start:end])
    except Exception:
        return None
@dataclass
class OpsTask:
    id:          str
    title:       str
    description: str = ""
    status:      str = "pending"
    result:      str = ""

class TaskStore:
    def __init__(self):
        self._tasks = {}; self._counter = 0
    def add(self, title, description=""):
        self._counter += 1
        tid = f"task_{self._counter:03d}"
        t = OpsTask(id=tid, title=title, description=description)
        self._tasks[tid] = t; return t
    def get(self, task_id): return self._tasks.get(task_id)
    def all(self): return list(self._tasks.values())
    def pending(self): return [t for t in self._tasks.values() if t.status == "pending"]
    def update(self, task_id, status, result=""):
        t = self._tasks.get(task_id)
        if t: t.status = status
        if t and result: t.result = result
        return t
    def __len__(self): return len(self._tasks)
def build_ops_tools(store, executor_fn=None):
    def list_tasks(status=None):
        tasks = store.all() if status is None else [t for t in store.all() if t.status == status]
        if not tasks: return "No tasks found."
        return "\n".join("[" + t.id + "] [" + t.status + "] " + t.title for t in tasks)
    def run_task(task_id):
        t = store.get(task_id)
        if t is None: return "Error: task " + repr(task_id) + " not found"
        if t.status not in ("pending", "failed"): return "Task " + task_id + " is already " + t.status
        store.update(task_id, "running")
        try:
            result = str(executor_fn(t)) if executor_fn else "Completed: " + t.title
        except Exception as exc:
            store.update(task_id, "failed", str(exc)); return "Error: " + str(exc)
        store.update(task_id, "done", result)
        return "Task " + task_id + " done: " + result
    def check_status(task_id):
        t = store.get(task_id)
        if t is None: return "Error: task " + repr(task_id) + " not found"
        return "[" + task_id + "] " + t.title + ": " + t.status + (" — " + t.result if t.result else "")
    def generate_report(scope="all"):
        tasks = store.all()
        if not tasks: return "No tasks in store."
        counts = {}
        for t in tasks: counts[t.status] = counts.get(t.status, 0) + 1
        summary = ", ".join(str(v) + " " + k for k, v in sorted(counts.items()))
        return "Ops Report (" + str(scope) + "): " + str(len(tasks)) + " tasks — " + summary
    return {"list_tasks": list_tasks, "run_task": run_task,
            "check_status": check_status, "generate_report": generate_report}

# ── Exercise: implement parse_ops_step, format_ops_step, run_ops_step ─────────

def _line_value(text, prefix):
    """Return the value after prefix on the first matching line (case-insensitive)."""
    prefix_lower = prefix.lower()
    for line in str(text).splitlines():
        stripped = line.strip()
        if stripped.lower().startswith(prefix_lower):
            return stripped[len(prefix):].strip()
    return ""


def parse_ops_step(text):
    """Parse one ReAct step.

    Returns {"thought": str, "action": str|None, "input": dict, "final": str|None}.
    Never raises.
    """
    # TODO:
    # 1. thought = _line_value(text, "Thought:")
    # 2. final   = _line_value(text, "Final Answer:")
    # 3. If final is non-empty: return {thought, action=None, input={}, final}
    # 4. action  = _line_value(text, "Action:")
    # 5. raw_input = _line_value(text, "Input:")
    # 6. args = safe_parse_json(raw_input) if raw_input else {}; if not dict: {}
    # 7. return {thought, action=action or None, input=args, final=None}
    return {"thought": "", "action": None, "input": {}, "final": None}


def format_ops_step(step, observation=""):
    """Format a parsed step + observation into a scratchpad string."""
    # TODO: build lines list:
    # - if step["thought"]: append "Thought: " + thought
    # - if step["final"]:   append "Final Answer: " + final
    # - elif step["action"]: append "Action: " + action, "Input: " + json.dumps(input)
    #   if observation: append "Observation: " + observation
    # return "\n".join(lines) + "\n"
    return ""


def build_ops_prompt(goal, tools, scratchpad=""):
    tool_names = "\n".join("  - " + n for n in tools)
    system = "\n".join([
        "You are an autonomous ops agent. Complete the goal using the available tools.",
        "Available tools:", tool_names, "",
        "Thought: <reasoning>  Action: <tool>  Input: <json>",
        "OR  Thought: <reasoning>  Final Answer: <summary>",
    ])
    user = "Goal: " + str(goal)
    if scratchpad: user += "\n\n" + scratchpad.rstrip()
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def run_ops_step(goal, tools, scratchpad="", llm_fn=None):
    """Call LLM once and parse the response into a step dict."""
    # TODO: call call_llm(build_ops_prompt(goal, tools, scratchpad), llm_fn=llm_fn)
    # then return parse_ops_step(response)
    return {"thought": "", "action": None, "input": {}, "final": None}


### Checks

In [ ]:
checks = 0

# 1 — parse_ops_step parses an action step correctly
try:
    text = 'Thought: I should list tasks\nAction: list_tasks\nInput: {"status": "pending"}'
    step = parse_ops_step(text)
    assert step["action"] == "list_tasks"
    assert step["input"] == {"status": "pending"}
    assert step["final"] is None
    assert "list tasks" in step["thought"].lower()
    checks += 1; print("✅ 1 parse_ops_step parses action step correctly")
except Exception as e:
    print("❌ 1:", e)

# 2 — parse_ops_step parses a final-answer step correctly
try:
    text = "Thought: All done\nFinal Answer: All tasks completed successfully."
    step = parse_ops_step(text)
    assert step["final"] == "All tasks completed successfully."
    assert step["action"] is None
    assert step["input"] == {}
    checks += 1; print("✅ 2 parse_ops_step parses final-answer step correctly")
except Exception as e:
    print("❌ 2:", e)

# 3 — parse_ops_step never raises on garbage input
try:
    for bad in ["", "random text", "Action: only", '{"json": "object"}']:
        result = parse_ops_step(bad)
        assert isinstance(result, dict)
        assert "action" in result and "final" in result
    checks += 1; print("✅ 3 parse_ops_step never raises on garbage input")
except Exception as e:
    print("❌ 3:", e)

# 4 — run_ops_step calls llm_fn and parses the result
try:
    mock_llm = lambda m: "Thought: checking\nAction: generate_report\nInput: {}"
    store = TaskStore(); store.add("T1")
    tools = build_ops_tools(store)
    step = run_ops_step("generate report", tools, llm_fn=mock_llm)
    assert step["action"] == "generate_report"
    assert step["final"] is None
    checks += 1; print("✅ 4 run_ops_step calls llm_fn and parses result")
except Exception as e:
    print("❌ 4:", e)

# 5 — format_ops_step produces a scratchpad fragment
try:
    step = {"thought": "I'll list tasks", "action": "list_tasks", "input": {}, "final": None}
    fragment = format_ops_step(step, observation="task_001 pending Task A")
    assert "Thought:" in fragment and "Action: list_tasks" in fragment
    assert "Observation:" in fragment
    checks += 1; print("✅ 5 format_ops_step produces readable scratchpad fragment")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
